# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Astroking2004/flyrank-ml-internship1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a ranking / scoring problem. The decision is which content pages deserve attention first, so the model should output a priority score that helps an editor decide where to spend limited refresh time.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import pandas as pd


# Look for the starter CSV in a few common locations.
def find_data_path():
    candidates = [
        Path("content_refresh_anonymized.csv"),
        Path.cwd() / "content_refresh_anonymized.csv",
        Path.cwd().parent / "content_refresh_anonymized.csv",
        Path.cwd().parent.parent / "content_refresh_anonymized.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Could not find the starter CSV.")


# Load the data once and create a consistent label column for later cells.
data_path = find_data_path()
df = pd.read_csv(data_path)

if "is_declining_label" in df.columns:
    df["is_declining_label"] = df["is_declining_label"].astype(int)
elif "trend_direction" in df.columns:
    df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)
else:
    raise KeyError("No decline label column found in the dataset.")

print("Rows:", len(df))
print("Observed decline rate:", f"{df['is_declining_label'].mean():.1%}")

Rows: 30000
Observed decline rate: 54.2%


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I would predict whether a page is likely to decline in the near term, using an observed later-time outcome: whether impressions in the most recent 30 days are lower than the previous 30 days. That makes the target an observed outcome rather than a hand-written rule.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd


df = pd.read_csv(data_path)

if "is_declining_label" in df.columns:
    df["is_declining_label"] = df["is_declining_label"].astype(int)
elif "trend_direction" in df.columns:
    df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)
else:
    raise KeyError("No decline label column found in the dataset.")

print("Positive label share:", f"{df['is_declining_label'].mean():.1%}")
print("Positive label count:", int(df["is_declining_label"].sum()))

Positive label share: 54.2%
Positive label count: 16262


## 3. Success metric

*One metric you can defend. What number means 'good'?*

A practical metric is precision@50. If the top 50 pages the model ranks are mostly true declines, the editor saves time and avoids wasting effort on low-priority pages. A strong model should beat the random baseline by a wide margin.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd


df = pd.read_csv(data_path)

if "is_declining_label" in df.columns:
    df["is_declining_label"] = df["is_declining_label"].astype(int)
elif "trend_direction" in df.columns:
    df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)
else:
    raise KeyError("No decline label column found in the dataset.")


top_k = 50
base_rate = df["is_declining_label"].mean()
expected_hits_at_random = int(round(base_rate * top_k))

print(f"Base rate of the target: {base_rate:.1%}")
print(f"Random precision@{top_k}: {base_rate:.1%}")
print(f"Expected hits in the top {top_k} by random chance: {expected_hits_at_random}")

Base rate of the target: 54.2%
Random precision@50: 54.2%
Expected hits in the top 50 by random chance: 27


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row is one content page. The dataframe mixes page metadata, content attributes, and 90-day search activity so that each row represents a single page that could be prioritized for refresh.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd


df = pd.read_csv(data_path)

if "is_declining_label" in df.columns:
    df["is_declining_label"] = df["is_declining_label"].astype(int)
elif "trend_direction" in df.columns:
    df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)
else:
    raise KeyError("No decline label column found in the dataset.")

cols = [
    "content_id",
    "content_type",
    "impressions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "is_declining_label",
]
sample = df[cols].head(5)

print("Shape:", df.shape)
print(sample.to_string(index=False))

Shape: (30000, 45)
          content_id    content_type  impressions_90d  impressions_last_30d  impressions_prev_30d  is_declining_label
content_304f48230142 keyword article             3803                   578                   987                   1
content_a1fb4e703a9e keyword article            15320                  2501                  5915                   1
content_9aa793d4d895 keyword article            12581                  2382                  6089                   1
content_331d6c4de07b keyword article            11751                  3626                  4206                   0
content_d99b7a2d90ca keyword article            19140                  4211                  6452                   1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule is too brittle because the decline pattern depends on many signals at once: content type, traffic scale, recent trend, and missing context. The same rule can miss pages that are actually declining and over-flag pages that look similar but behave differently.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd


df = pd.read_csv(data_path)

if "is_declining_label" in df.columns:
    df["is_declining_label"] = df["is_declining_label"].astype(int)
elif "trend_direction" in df.columns:
    df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)
else:
    raise KeyError("No decline label column found in the dataset.")

by_type = (
    df.groupby("content_type")["is_declining_label"]
      .mean()
      .sort_values(ascending=False)
)

print("Decline rate by content type:")
print(by_type.round(3).to_string())

Decline rate by content type:
content_type
comparison article    0.572
keyword article       0.561
feedly article        0.287


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.